# UD2.06. De cuaderno a aplicación: Streamlit

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Proyecto PR2**

Un cuaderno demuestra que algo funciona. Una aplicación permite que **otra persona lo use**. Este
cuaderno es el puente entre las dos cosas.

Streamlit convierte un script de Python en una aplicación web sin escribir HTML ni JavaScript. Y
tiene una peculiaridad que hay que entender antes de escribir nada, porque explica casi todos los
comportamientos raros del principio.

> **Streamlit no se ejecuta dentro del cuaderno.** Aquí se escriben los ficheros con
> `%%writefile` y se ejecutan desde el terminal con `streamlit run`. Las celdas de este cuaderno
> generan ficheros; no producen interfaz.

In [ ]:
!pip install -q streamlit requests python-dotenv

## 1. El modelo de ejecución

> **Cada vez que el usuario toca algo, Streamlit vuelve a ejecutar el script entero, de arriba
> abajo.**

No hay eventos ni retrollamadas. Escribes el script como si fuera de un solo uso, y Streamlit lo
reejecuta cada vez que cambia cualquier control.

Dos consecuencias, y las dos cuestan dinero o tiempo si no se saben:

1. **Lo caro se ejecuta en cada interacción.** Incluidas las llamadas al API que pagas. Sin
   caché, mover un deslizador es una llamada nueva.
2. **Las variables normales se pierden** entre ejecuciones. Lo que debe sobrevivir va en
   `st.session_state`.

Vamos a verlo en una aplicación mínima antes de creerlo.

In [ ]:
%%writefile demo_ejecucion.py
"""Demostración del modelo de reejecución de Streamlit.

Ejecuta:  streamlit run demo_ejecucion.py
Mueve el deslizador y mira los dos contadores.
"""
import time

import streamlit as st

st.title("Cómo se ejecuta una aplicación de Streamlit")

# Esta variable se crea de nuevo en cada ejecución del script
contador_normal = 0
contador_normal += 1

# Esta sobrevive porque vive en session_state
if "ejecuciones" not in st.session_state:
    st.session_state.ejecuciones = 0
st.session_state.ejecuciones += 1

valor = st.slider("Mueve esto", 0, 100, 50)

st.metric("Variable normal", contador_normal)
st.metric("En session_state", st.session_state.ejecuciones)

st.write(f"Hora de esta ejecución: {time.strftime('%H:%M:%S')}")
st.caption(
    "La variable normal siempre vale 1: el script se ha vuelto a ejecutar entero. "
    "La de session_state cuenta cuántas veces ha pasado."
)

```bash
streamlit run demo_ejecucion.py
```

Mueve el deslizador varias veces. El primer contador se queda en 1 y el segundo sube. Esa es toda
la diferencia, y explica por qué una variable donde guardabas el historial aparece vacía.

## 2. Los componentes que vas a usar

Tres familias: **entrada**, **acción** y **salida**.

In [ ]:
%%writefile demo_componentes.py
"""Catálogo de los componentes de Streamlit que usa el proyecto de la unidad."""
import streamlit as st

st.set_page_config(page_title="Componentes", page_icon=None, layout="centered")

st.title("Analizador de texto")
st.caption("Ejemplo de estructura. No llama a ningún servicio.")

# --- Entrada ---
st.header("Entrada")
texto = st.text_area("Pega aquí tu texto", height=140,
                     placeholder="Escribe o pega el texto que quieras analizar")
columna_a, columna_b = st.columns(2)
with columna_a:
    idioma = st.selectbox("Idioma", ["es", "ca", "en"], index=0)
with columna_b:
    umbral = st.slider("Confianza mínima", 0.0, 1.0, 0.5, 0.05)

fichero = st.file_uploader("O sube un fichero de texto", type=["txt"])

# --- Acción ---
if st.button("Analizar", type="primary", disabled=not (texto or fichero)):
    with st.spinner("Consultando el servicio..."):
        resultado = {
            "sentimiento": "positivo",
            "confianza": 0.93,
            "frases_clave": ["atención al ciudadano", "trámite rápido"],
        }

    # --- Salida ---
    st.header("Resultado")
    izquierda, derecha = st.columns([1, 2])
    with izquierda:
        st.metric("Sentimiento", resultado["sentimiento"].capitalize(),
                  f"{resultado['confianza']:.0%} de confianza")
    with derecha:
        st.write("**Frases clave**")
        for frase in resultado["frases_clave"]:
            st.write("-", frase)

    with st.expander("Respuesta completa del servicio"):
        st.json(resultado)

    st.success("Análisis completado")

Fíjate en el `disabled=not (texto or fichero)` del botón. Un botón que no puede hacer nada debe
estar desactivado, no fallar cuando se pulsa. Es de lo más barato que se puede hacer por la
usabilidad de una interfaz.

Y en el `st.expander` del final: **los datos completos, disponibles pero plegados**. Quien quiere
el JSON lo tiene; quien no, no lo ve.

## 3. Mostrar JSON en crudo no es una interfaz

Un servicio de IA devuelve una estructura anidada con quince campos. La aplicación tiene que
decidir cuáles importan y presentarlos, no volcar el JSON en pantalla.

Cuatro reglas, que están en la rúbrica de PR2:

- **El dato clave, primero y grande.** `st.metric()` para el resultado principal.
- **La confianza, siempre visible.** Un "positivo" al 51 % no es lo mismo que al 98 %, y ocultar
  el número induce a error.
- **El color, nunca como único canal.** Quien no distingue el rojo del verde debe poder leer el
  resultado igual. Acompaña siempre el color de texto.
- **Los datos completos, plegados.**

In [ ]:
%%writefile demo_presentacion.py
"""Dos formas de presentar el mismo resultado: la mala y la correcta."""
import streamlit as st

RESPUESTA = {
    "sentiment": "negative",
    "confidenceScores": {"positive": 0.02, "neutral": 0.11, "negative": 0.87},
    "sentences": [{"text": "Llevo tres semanas esperando.", "sentiment": "negative"}],
}

st.title("Presentar un resultado")

st.subheader("Así no")
st.json(RESPUESTA)
st.caption("Correcto y completo. E inútil para quien no sabe qué es confidenceScores.")

st.divider()

st.subheader("Así sí")

ETIQUETAS = {"positive": "Positivo", "neutral": "Neutro", "negative": "Negativo"}
SIMBOLOS = {"positive": "+", "neutral": "=", "negative": "-"}

etiqueta = RESPUESTA["sentiment"]
confianza = RESPUESTA["confidenceScores"][etiqueta]

st.metric(
    "Sentimiento detectado",
    f"{SIMBOLOS[etiqueta]} {ETIQUETAS[etiqueta]}",
    f"{confianza:.0%} de confianza",
)

# El color refuerza, pero el texto ya lo dice todo por sí solo
mensaje = f"{ETIQUETAS[etiqueta]} con un {confianza:.0%} de confianza"
if etiqueta == "negative":
    st.error(mensaje)
elif etiqueta == "positive":
    st.success(mensaje)
else:
    st.info(mensaje)

st.write("**Reparto de las puntuaciones**")
st.bar_chart({ETIQUETAS[k]: [v] for k, v in RESPUESTA["confidenceScores"].items()})

if confianza < 0.6:
    st.warning(
        "La confianza es baja. Este resultado no debería usarse para tomar una decisión "
        "sin revisión humana."
    )

with st.expander("Respuesta completa del servicio"):
    st.json(RESPUESTA)

El aviso de confianza baja del final no es adorno. Si tu aplicación va a influir en una decisión
que afecta a una persona, decir "no me fío de este resultado" es parte de hacerlo bien, y está en
la rúbrica.

## 4. La estructura del proyecto, y la regla que hay detrás

```
mi_app_ia/
├─ app/
│  ├─ main.py                 punto de entrada
│  ├─ pages/                  Streamlit las descubre solo, por orden alfabético
│  │   ├─ 1_Texto.py
│  │   ├─ 2_Imagen.py
│  │   └─ 3_Voz.py
│  ├─ servicios/              un módulo por servicio de IA
│  │   ├─ lenguaje.py
│  │   ├─ vision.py
│  │   └─ voz.py
│  └─ utils/http.py           sesión, reintentos, errores
├─ .streamlit/secrets.toml    NO se sube
├─ .env.example               SÍ se sube
├─ requirements.txt
└─ README.md
```

> **La interfaz no llama al API directamente.**

`main.py` llama a `servicios/lenguaje.py`, y ese módulo es el único que sabe de URL, claves y
JSON. Tres cosas que se ganan con eso:

1. Cambias de proveedor tocando un fichero.
2. Puedes probar los servicios sin levantar la interfaz, que es lo que hiciste en los cuadernos
   03, 04 y 05.
3. Los mensajes de error los decide la interfaz, no el módulo. El mismo fallo se cuenta distinto
   a un usuario que a un log.

Es la regla estructural que se corrige en P2.2 y en PR2.

## 5. Una aplicación completa, con todo lo anterior

Con los módulos de los cuadernos anteriores puestos en `servicios/`, la página queda así de
corta. Eso es la señal de que la separación está bien hecha.

In [ ]:
%%writefile app_texto.py
"""Analizador de texto. Ejemplo completo de página de Streamlit.

Espera un módulo servicios/lenguaje.py como el del cuaderno UD2.03.
Ejecuta:  streamlit run app_texto.py
"""
import streamlit as st

from servicios import lenguaje
from servicios.http import (ErrorAutenticacion, ErrorCuota, ErrorPeticion,
                            ErrorProveedor, ErrorServicio)

st.set_page_config(page_title="Analizador de texto", layout="centered")

st.title("Analizador de texto")
st.caption(
    "Esta aplicación envía el texto que escribas a Azure AI Language, un servicio de "
    "Microsoft alojado en la Unión Europea, para analizarlo. No se guarda nada."
)

texto = st.text_area("Texto a analizar", height=160)
idioma = st.selectbox("Idioma del texto", ["es", "ca", "en"])

if st.button("Analizar", type="primary", disabled=not texto.strip()):
    try:
        with st.spinner("Consultando el servicio..."):
            sentimiento = lenguaje.analiza_sentimiento([texto], idioma)[0]
            frases = lenguaje.extrae_frases_clave([texto], idioma)[0]

    except ErrorAutenticacion:
        st.error("La aplicación no está bien configurada. Avisa a quien la mantiene.")
    except ErrorCuota:
        st.warning("Se ha superado el límite de peticiones. Espera unos segundos e inténtalo otra vez.")
    except ErrorPeticion as error:
        st.error(f"El texto no se ha podido procesar: {error}")
    except ErrorProveedor:
        st.error("El servicio no está disponible ahora mismo. Vuelve a intentarlo en unos minutos.")
    except ErrorServicio:
        st.error("Ha ocurrido un problema al analizar el texto.")

    else:
        etiqueta = sentimiento["sentimiento"]
        confianza = max(sentimiento["positivo"], sentimiento["neutro"], sentimiento["negativo"])

        st.metric("Sentimiento", etiqueta.capitalize(), f"{confianza:.0%} de confianza")

        if frases:
            st.write("**Frases clave**")
            st.write(", ".join(frases))
        else:
            st.info("No se han encontrado frases clave en este texto.")

        if confianza < 0.6:
            st.warning("Confianza baja: convendría revisar este resultado a mano.")

        with st.expander("Detalle del análisis"):
            st.json(sentimiento)

Mira la escalera de `except`. **Cada causa tiene su mensaje**, y ninguno de ellos enseña una traza
ni un código HTTP:

| Excepción | Qué le decimos al usuario | Por qué |
|---|---|---|
| `ErrorAutenticacion` | La aplicación está mal configurada | No es culpa suya y no puede arreglarlo |
| `ErrorCuota` | Espera unos segundos | Sí puede hacer algo: esperar |
| `ErrorPeticion` | El texto no se ha podido procesar | El problema está en la entrada |
| `ErrorProveedor` | El servicio no está disponible | Vuelve más tarde |

"Ha ocurrido un error" no ayuda a nadie. Distinguir es barato y se corrige.

Y el `else` del `try`: el código del caso correcto va ahí, no dentro del `try`. Así se ve de un
vistazo qué parte puede fallar y qué parte no.

## 6. Estado que sobrevive: `st.session_state`

Todo lo que deba durar más de una interacción vive ahí: un historial, el resultado que ya has
calculado, el paso en el que va un asistente de varias pantallas.

In [ ]:
%%writefile demo_historial.py
"""Historial de análisis que sobrevive a las reejecuciones."""
import streamlit as st

st.title("Historial")

if "historial" not in st.session_state:
    st.session_state.historial = []

texto = st.text_input("Escribe algo y pulsa Añadir")

columna_a, columna_b = st.columns(2)
with columna_a:
    if st.button("Añadir", disabled=not texto.strip()):
        st.session_state.historial.append(texto)
with columna_b:
    if st.button("Vaciar", disabled=not st.session_state.historial):
        st.session_state.historial = []

st.write(f"**{len(st.session_state.historial)} entradas**")
for i, entrada in enumerate(reversed(st.session_state.historial), start=1):
    st.write(f"{i}. {entrada}")

if st.session_state.historial:
    st.download_button(
        "Descargar historial",
        data="\n".join(st.session_state.historial),
        file_name="historial.txt",
    )

`st.session_state` es **por sesión de navegador**. Si dos personas abren la aplicación, cada una
tiene el suyo. Y se pierde al recargar la página: no es almacenamiento, es memoria de la sesión.
Lo que deba persistir de verdad va a un fichero o a una base de datos.

## 7. Antes de pasar al despliegue

Prueba las cuatro aplicaciones de este cuaderno:

```bash
streamlit run demo_ejecucion.py
streamlit run demo_componentes.py
streamlit run demo_presentacion.py
streamlit run demo_historial.py
```

Y comprueba estas cuatro cosas en la tuya, que son las que se miran en PR2:

1. Con el campo vacío, el botón está desactivado y no se rompe nada.
2. Con las credenciales mal, sale un mensaje en castellano y no una traza.
3. El resultado principal se ve sin desplegar nada, y la confianza está a la vista.
4. Hay un aviso visible de que la aplicación usa IA y de dónde se procesan los datos.

Ese cuarto punto no es cortesía: el Reglamento Europeo de IA exige transparencia en los sistemas
que interactúan con personas.

Siguiente: **UD2.07**, robustez, caché y despliegue. Ahí es donde esta aplicación deja de costar
dinero cada vez que alguien mueve un control.